# **10_paired_analysis_for_vit_bg_suppression**

In [ ]:
# ============================================================
# 10_paired_analysis_for_vit_bg_suppression
# Standalone paired analysis after runtime disconnect
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/MLP/Project5")
ANALYSIS_OUTPUTS = PROJECT_ROOT / "analysis_outputs"
ANALYSIS_OUTPUTS.mkdir(parents=True, exist_ok=True)

EXP_NAME = "vit_b16_bg_suppression_patch_lambda0001_expand_10ep"
EXP_DIR = PROJECT_ROOT / "final_outputs" / EXP_NAME
EVAL_DIR = EXP_DIR / "eval_jpeg_q30_controlled"

orig_path = EVAL_DIR / "predictions_original.csv"
jpeg_path = EVAL_DIR / "predictions_jpeg_q30_controlled.csv"
summary_path = EVAL_DIR / "summary_jpeg_q30_controlled.csv"

print("orig exists:", orig_path.exists(), orig_path)
print("jpeg exists:", jpeg_path.exists(), jpeg_path)
print("summary exists:", summary_path.exists(), summary_path)

# **11_load_predictions**

In [ ]:
# ============================================================
# 11_load_predictions
# ============================================================

orig = pd.read_csv(orig_path)
jpeg = pd.read_csv(jpeg_path)

print("orig shape:", orig.shape)
print("jpeg shape:", jpeg.shape)

print("orig columns:")
print(orig.columns.tolist())

display(orig.head())
display(jpeg.head())

summary_df = pd.read_csv(summary_path)
display(summary_df)

# **12_build_paired_table**

In [ ]:
# ============================================================
# 12_build_paired_table
# Build paired original-vs-JPEG table
# ============================================================

def add_pair_key(df):
    df = df.copy()
    df["pair_key"] = (
        df["true_label_name"].astype(str)
        + "_"
        + df["image_id"].astype(str)
    )
    return df

orig = add_pair_key(orig)
jpeg = add_pair_key(jpeg)

print("Original duplicated pair_key:", orig["pair_key"].duplicated().sum())
print("JPEG duplicated pair_key:", jpeg["pair_key"].duplicated().sum())

orig_small = orig[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "file_extension",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

jpeg_small = jpeg[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "file_extension",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

paired = orig_small.merge(
    jpeg_small,
    on="pair_key",
    suffixes=("_orig", "_jpeg"),
    how="inner"
)

label_mismatch = (
    paired["true_label_name_orig"] != paired["true_label_name_jpeg"]
).sum()

print("label mismatch:", label_mismatch)
print("paired shape:", paired.shape)

paired["prob_fake_drop"] = paired["prob_fake_orig"] - paired["prob_fake_jpeg"]
paired["prob_fake_change"] = paired["prob_fake_jpeg"] - paired["prob_fake_orig"]
paired["prob_real_change"] = paired["prob_real_jpeg"] - paired["prob_real_orig"]

display(paired.head())

# **13_transition_analysis**

In [ ]:
# ============================================================
# 13_transition_analysis
# Classify original -> JPEG prediction transitions
# ============================================================

def classify_transition(row):
    true_label = row["true_label_name_orig"]
    orig_pred = row["pred_label_name_orig"]
    jpeg_pred = row["pred_label_name_jpeg"]

    if true_label == "fake":
        if orig_pred == "fake" and jpeg_pred == "fake":
            return "fake_TP_to_TP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "fake_TP_to_FN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "fake_FN_to_TP"
        elif orig_pred == "real" and jpeg_pred == "real":
            return "fake_FN_to_FN"

    if true_label == "real":
        if orig_pred == "real" and jpeg_pred == "real":
            return "real_TN_to_TN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "real_TN_to_FP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "real_FP_to_TN"
        elif orig_pred == "fake" and jpeg_pred == "fake":
            return "real_FP_to_FP"

    return "other"

paired["transition"] = paired.apply(classify_transition, axis=1)

transition_counts = paired["transition"].value_counts().reset_index()
transition_counts.columns = ["transition", "count"]

display(transition_counts)

paired_save_path = ANALYSIS_OUTPUTS / "vit_b16_bg_suppression_patch_lambda0001_paired_predictions_original_vs_jpeg.csv"
paired.to_csv(paired_save_path, index=False, encoding="utf-8-sig")

print("Saved paired predictions:", paired_save_path)

# **14_transition_summary**

In [ ]:
# ============================================================
# 14_transition_summary
# Save compact transition summary
# ============================================================

fake_subset = paired[paired["true_label_name_orig"] == "fake"].copy()
real_subset = paired[paired["true_label_name_orig"] == "real"].copy()

count_dict = paired["transition"].value_counts().to_dict()

bg_transition_summary = pd.DataFrame([
    {
        "model_key": "vit_b16_bg_suppression_patch_lambda0001",
        "model_name": "ViT-B/16 + Patch-BG-Suppression",
        "method_name": "Patch-level Background Suppression",
        "lambda_bg": 0.001,
        "fake_total": len(fake_subset),
        "real_total": len(real_subset),

        "fake_TP_to_TP": count_dict.get("fake_TP_to_TP", 0),
        "fake_TP_to_FN": count_dict.get("fake_TP_to_FN", 0),
        "fake_FN_to_TP": count_dict.get("fake_FN_to_TP", 0),
        "fake_FN_to_FN": count_dict.get("fake_FN_to_FN", 0),

        "real_TN_to_TN": count_dict.get("real_TN_to_TN", 0),
        "real_TN_to_FP": count_dict.get("real_TN_to_FP", 0),
        "real_FP_to_TN": count_dict.get("real_FP_to_TN", 0),
        "real_FP_to_FP": count_dict.get("real_FP_to_FP", 0),

        "fake_TP_to_FN_ratio_percent": count_dict.get("fake_TP_to_FN", 0) / len(fake_subset) * 100,

        "mean_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].std(),

        "mean_prob_fake_drop_real_images": real_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_real_images": real_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_real_images": real_subset["prob_fake_drop"].std(),
    }
])

summary_save_path = ANALYSIS_OUTPUTS / "vit_b16_bg_suppression_patch_lambda0001_transition_summary_original_vs_jpeg.csv"
bg_transition_summary.to_csv(summary_save_path, index=False, encoding="utf-8-sig")

print("Saved:", summary_save_path)
display(bg_transition_summary)

# **15_compare_vit_three_methods**

In [ ]:
# ============================================================
# 15_compare_vit_three_methods
# Compare ViT baseline vs LazyStrike vs Patch-BG-Suppression
# ============================================================

# Known baseline / LazyStrike values from previous analyses
vit_three_compare = pd.DataFrame([
    {
        "model_key": "vit_b16",
        "model_name": "ViT-B/16",
        "method_name": "Baseline",
        "fake_total": 6000,
        "fake_TP_to_FN": 102,
        "fake_FN_to_TP": 19,
        "fake_TP_to_FN_ratio_percent": 102 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.016788,
    },
    {
        "model_key": "vit_b16_lazystrike_k1",
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "method_name": "LazyStrike-k1",
        "fake_total": 6000,
        "fake_TP_to_FN": 136,
        "fake_FN_to_TP": 12,
        "fake_TP_to_FN_ratio_percent": 136 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.019980,
    },
    {
        "model_key": "vit_b16_bg_suppression_patch_lambda0001",
        "model_name": "ViT-B/16 + Patch-BG-Suppression",
        "method_name": "Patch-level Background Suppression",
        "fake_total": int(bg_transition_summary.loc[0, "fake_total"]),
        "fake_TP_to_FN": int(bg_transition_summary.loc[0, "fake_TP_to_FN"]),
        "fake_FN_to_TP": int(bg_transition_summary.loc[0, "fake_FN_to_TP"]),
        "fake_TP_to_FN_ratio_percent": float(bg_transition_summary.loc[0, "fake_TP_to_FN_ratio_percent"]),
        "mean_prob_fake_drop_fake_images": float(bg_transition_summary.loc[0, "mean_prob_fake_drop_fake_images"]),
    },
])

save_path = ANALYSIS_OUTPUTS / "vit_baseline_vs_lazystrike_vs_bg_suppression_paired_comparison.csv"
vit_three_compare.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(vit_three_compare)

# **16_compare_summary_metrics**

In [ ]:
# ============================================================
# 16_compare_summary_metrics
# Compare original/JPEG metrics for ViT baseline, LazyStrike, and BG suppression
# ============================================================

bg_summary = pd.read_csv(summary_path).iloc[0].to_dict()

vit_summary_compare = pd.DataFrame([
    {
        "model_name": "ViT-B/16",
        "method_name": "Baseline",
        "best_epoch": None,
        "best_val_f1": None,
        "original_accuracy": 0.9471,
        "original_recall_fake": 0.9378,
        "original_f1_fake": 0.9466,
        "jpeg_accuracy": 0.9425,
        "jpeg_recall_fake": 0.9240,
        "jpeg_f1_fake": 0.9428,
        "recall_fake_drop": 0.0138,
        "f1_fake_drop": 0.0038,
    },
    {
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "method_name": "LazyStrike-k1",
        "best_epoch": 7,
        "best_val_f1": 0.949902,
        "original_accuracy": 0.942441,
        "original_recall_fake": 0.956333,
        "original_f1_fake": 0.943207,
        "jpeg_accuracy": 0.939608,
        "jpeg_recall_fake": 0.935667,
        "jpeg_f1_fake": 0.939346,
        "recall_fake_drop": 0.020667,
        "f1_fake_drop": 0.003861,
    },
    {
        "model_name": "ViT-B/16 + Patch-BG-Suppression",
        "method_name": "Patch-level Background Suppression",
        "best_epoch": bg_summary["best_epoch"],
        "best_val_f1": bg_summary["best_val_f1"],
        "original_accuracy": bg_summary["original_accuracy"],
        "original_recall_fake": bg_summary["original_recall_fake"],
        "original_f1_fake": bg_summary["original_f1_fake"],
        "jpeg_accuracy": bg_summary["jpeg_accuracy"],
        "jpeg_recall_fake": bg_summary["jpeg_recall_fake"],
        "jpeg_f1_fake": bg_summary["jpeg_f1_fake"],
        "recall_fake_drop": bg_summary["recall_fake_drop"],
        "f1_fake_drop": bg_summary["f1_fake_drop"],
    },
])

save_path = ANALYSIS_OUTPUTS / "vit_baseline_vs_lazystrike_vs_bg_suppression_summary_metrics.csv"
vit_summary_compare.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(vit_summary_compare)